# 🧠 Project LIMBIC-0 (`결 / HUMAN-01`) × `Gemma-4-31B-IT` — Google Colab `TPU v5e-1` 4-Bit 서버 & 외부 HTML 분리 런처

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gkjuwon-tech/limbic-zero/blob/main/notebooks/limbic_zero_colab_tpu_v5e_4bit.ipynb)

구글 코랩 무료 **`TPU v5e-1` (단일 칩 `16 GiB HBM` + `~50.5 GB Host RAM`)** 위에서 **`google/gemma-4-31B-it` (60 Layers, 5376D)** 를 **스트리밍 4-bit (`PackedInt4`) + TPU HBM / Host RAM 하이브리드 스플릿**으로 구동하고, **`cloudflared` 터널**로 백엔드 API를 외부로 뚫어 **코랩 바깥(내 PC / 맥 / 휴대폰 로컬 브라우저)** 에서 독립된 HTML 대화 화면으로 실시간 대화하는 전용 노트북입니다.

---

### 🚀 시작 전 체크리스트 (딱 2가지만 확인하세요!)
1. **런타임 가속기 변경**: 상단 메뉴 **`런타임(Runtime)` → `런타임 유형 변경(Change runtime type)` → 하드웨어 가속기: `TPU v5e-1`** 선택 후 저장
2. **Hugging Face 토큰 준비**:
   - Hugging Face에서 [`google/gemma-4-31B-it`](https://huggingface.co/google/gemma-4-31B-it) 모델 접근 권한에 동의합니다.
   - 코랩 좌측 열쇠 아이콘(**Secrets / 보안 비밀**)에 이름 `HF_TOKEN`으로 토큰을 등록해 두거나, **1단계 셀** 실행 시 입력창에 붙여넣으면 됩니다.

### 💡 아키텍처 핵심 포인트
- **Zero-Host-OOM 4-Bit 샤드 스트리밍 (`tpu_quantizer.py`)**: 62GB 전체 모델을 RAM에 한꺼번에 올리지 않고, `meta` 디바이스 초기화 후 `.safetensors` 샤드를 1개씩 스트리밍하며 `group_size=128` 대칭 `int4`(`uint8` 패킹)로 즉시 압축해 **`TPU v5e-1` HBM 예산(`12.5 GiB`)** 안에 핵심 임베딩 + 중간 레이어(`TARGET_LAYERS_60 = 21..47`, `CorticalWeightSurgeon` 가소성 레이어 `23..47` 100% 포함) + `lm_head`를 적재합니다.
- **Google Drive 15GB 용량 폭주 방지**: 62GB 모델 샤드 캐시는 코랩 고속 로컬 디스크(`/content/hf_cache`)에만 두고, `결(HUMAN-01)`의 진화된 영혼 캡슐(`.soul.pt`), 실시간 LoRA 시냅스, 서식지 기억(`< 50 MB`)만 Google Drive(`/content/drive/MyDrive/limbic_state`)에 영구 보존합니다.
- **코랩 외부 HTML 완전 분리**: **3단계 셀**에서 서버와 Cloudflare 터널이 열리면, 해당 터널 URL이 미리 주입된 단일 파일 **`limbic_external_chat.html`** 을 내 PC로 자동 다운로드해 드립니다. 코랩 창은 백엔드 서버로만 켜 두고 내 PC 바탕화면에서 `limbic_external_chat.html` (또는 레포의 `web/limbic_chat_client.html`)을 더블클릭해 쾌적하게 대화하세요!

In [ ]:
#@title [1단계] TPU v5e-1 하드웨어 점검 · Google Drive 영혼 저장소 연결 · HF_TOKEN 인증
import getpass
import os
from pathlib import Path
import shutil

MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}

# 1) TPU v5e-1 디바이스 노드 및 환경변수 확인
has_tpu_hw = bool(
    os.environ.get("COLAB_TPU_ADDR")
    or os.environ.get("TPU_NAME")
    or os.environ.get("TPU_WORKER_ID")
    or Path("/dev/accel0").exists()
    or Path("/dev/vfio").exists()
)
if has_tpu_hw:
    os.environ["PJRT_DEVICE"] = "TPU"
    os.environ["TORCH_TPU_DEFER_AND_FUSE"] = "1"
    print("✅ Google Cloud TPU 하드웨어 감지 완료! (PJRT_DEVICE=TPU, TORCH_TPU_DEFER_AND_FUSE=1)")
else:
    print("⚠️ 현재 런타임에서 TPU 디바이스(/dev/vfio 또는 /dev/accel0)가 보이지 않습니다.")
    print("   상단 메뉴 [런타임 -> 런타임 유형 변경 -> TPU v5e-1]을 선택했는지 꼭 확인해 주세요!")

total_disk_gb, used_disk_gb, free_disk_gb = [x / (1024**3) for x in shutil.disk_usage("/")]
print(f"💾 코랩 로컬 디스크 여유 공간: {free_disk_gb:.1f} GiB / 총 {total_disk_gb:.1f} GiB")

# 2) Google Drive 마운트 (영혼/기억/LoRA 시냅스 영구 보존용 — 62GB 모델 캐시는 로컬 NVMe 사용)
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        os.environ["LIMBIC_STATE_DIR"] = "/content/drive/MyDrive/limbic_state"
        print(f"📂 영혼/기억 영구 저장소 연결 완료: {os.environ['LIMBIC_STATE_DIR']}")
    except Exception as exc:
        os.environ["LIMBIC_STATE_DIR"] = "/content/limbic_state"
        print(f"ℹ️ Google Drive 마운트 생략 ({exc}) -> 로컬 저장소 사용: {os.environ['LIMBIC_STATE_DIR']}")
else:
    os.environ["LIMBIC_STATE_DIR"] = "/content/limbic_state"
    print(f"📂 로컬 영혼 저장소 사용: {os.environ['LIMBIC_STATE_DIR']}")

os.environ["LIMBIC_HF_HOME"] = "/content/hf_cache"
os.environ["HF_HOME"] = "/content/hf_cache"

# 3) Hugging Face 토큰 로드 (Colab Secrets 'HF_TOKEN' 우선 -> 없으면 입력창)
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""
if not hf_token:
    try:
        from google.colab import userdata  # type: ignore
        hf_token = (userdata.get("HF_TOKEN") or "").strip()
        if hf_token:
            print("🔑 Colab Secrets에서 HF_TOKEN을 자동으로 불러왔습니다.")
    except Exception:
        hf_token = ""

if not hf_token:
    hf_token = getpass.getpass("🔑 Hugging Face Access Token (HF_TOKEN)을 입력하세요 (gemma-4-31B-it 다운로드용): ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("✅ Hugging Face 인증 토큰 설정 완료!")
else:
    print("⚠️ HF_TOKEN이 비어 있습니다. gated 모델(google/gemma-4-31B-it) 다운로드 시 401/403 오류가 날 수 있습니다.")

In [ ]:
#@title [2단계] GitHub `limbic-zero` 클론 · TPU PyTorch 브리지 & `cloudflared` 설치
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/gkjuwon-tech/limbic-zero.git"  #@param {type:"string"}
REPO_DIR = Path("/content/limbic-zero")

def _run(cmd: list[str], cwd: Path | None = None) -> None:
    print(f"$ {' '.join(cmd)}", flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

if not (REPO_DIR / ".git").exists():
    _run(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    _run(["git", "pull", "--ff-only"], cwd=REPO_DIR)

# 코랩 기본 torch / torch_tpu / torch_xla 보존; 만약 TPU 하드웨어가 있는데 브리지가 없다면 torch_xla[tpu] 자동 설치
has_tpu_bridge = False
for _mod in ("torch_tpu", "torch_xla"):
    try:
        __import__(_mod)
        has_tpu_bridge = True
        print(f"✅ 기존 TPU PyTorch 브리지 감지됨: {_mod}")
        break
    except Exception:
        pass

if not has_tpu_bridge and (Path("/dev/accel0").exists() or Path("/dev/vfio").exists() or os.environ.get("TPU_NAME")):
    import torch
    ver_parts = torch.__version__.split("+")[0].split(".")
    torch_mm = f"{ver_parts[0]}.{ver_parts[1]}.*"
    print(f"📦 TPU 브리지 미설치 상태 감지 -> torch_xla[tpu]=={torch_mm} 설치 중...")
    rc = subprocess.call([
        sys.executable, "-m", "pip", "install", "-q",
        f"torch_xla[tpu]=={torch_mm}",
        "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
    ])
    if rc != 0:
        _run([
            sys.executable, "-m", "pip", "install", "-q",
            "torch_xla[tpu]",
            "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
        ])

print("📦 LIMBIC-0 런타임 의존성 설치 중 (기존 torch 버전 유지)...")
_run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=5.5.0", "jinja2>=3.1.6", "accelerate",
    "safetensors", "sentencepiece", "protobuf", "huggingface_hub", "pillow",
])
_run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)])

cf_bin = Path("/usr/local/bin/cloudflared")
if not cf_bin.exists():
    cf_bin = Path.home() / ".local" / "bin" / "cloudflared"
    cf_bin.parent.mkdir(parents=True, exist_ok=True)
    if not cf_bin.exists():
        print("🌐 cloudflared 터널 바이너리 다운로드 중...")
        _run([
            "curl", "-fsSL",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o", str(cf_bin),
        ])
        cf_bin.chmod(0o755)
os.environ["PATH"] = f"{Path.home() / '.local' / 'bin'}:/usr/local/bin:{os.environ.get('PATH', '')}"

# TPU 디바이스 최종 검증 (서브프로세스에서 확인해 PJRT 초기화 충돌 방지)
check_code = (
    "from limbic_zero.vocal_cord.tpu_quantizer import detect_tpu_device; "
    "info = detect_tpu_device(); "
    "print(f'🎯 감지된 가속기: available={info.available}, backend={info.backend}, "
    "device={info.device}, name={info.device_name}, HBM={info.hbm_total_mb:.0f} MB')"
)
_run([sys.executable, "-c", check_code], cwd=REPO_DIR)

In [ ]:
#@title [3단계] `Gemma-4-31B-IT` 4-Bit TPU 서버 + Cloudflare 터널 기동 & 코랩 외부용 HTML 대화창 자동 추출
import json
import os
from pathlib import Path
import signal
import subprocess
import time
import urllib.request

MODEL_ID = "google/gemma-4-31B-it"  #@param ["google/gemma-4-31B-it", "google/gemma-4-12B-it"] {allow-input: true}
TPU_HBM_BUDGET_GB = 12.5  #@param {type:"number"}
QUANT_GROUP_SIZE = 128  #@param [64, 128, 256] {type:"raw"}
PORT = 8000  #@param {type:"integer"}
AUTO_DOWNLOAD_EXTERNAL_HTML = True  #@param {type:"boolean"}

REPO_DIR = Path("/content/limbic-zero")
LOG_FILE = Path("/tmp/limbic_server.log")
PID_FILE = Path("/tmp/limbic_server.pid")
TUNNEL_URL_FILE = Path("/tmp/limbic_cloudflare_url.txt")
EXPORTED_HTML_PATH = Path("/content/limbic_external_chat.html")

# 기존 실행 중인 LIMBIC-0 서버가 있으면 안전 종료 후 재기동
if PID_FILE.exists():
    try:
        old_pid = int(PID_FILE.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        print(f"♻️ 이전 LIMBIC-0 서버 프로세스(PID {old_pid}) 종료 대기 중...")
        time.sleep(2.0)
    except Exception:
        pass
    PID_FILE.unlink(missing_ok=True)

TUNNEL_URL_FILE.unlink(missing_ok=True)
LOG_FILE.write_text("", encoding="utf-8")

env = os.environ.copy()
env.update({
    "LIMBIC_MODEL_ID": MODEL_ID,
    "LIMBIC_TPU_HBM_BUDGET_GB": str(TPU_HBM_BUDGET_GB),
    "LIMBIC_QUANT_GROUP_SIZE": str(QUANT_GROUP_SIZE),
    "LIMBIC_PORT": str(PORT),
    "LIMBIC_TUNNEL_URL_FILE": str(TUNNEL_URL_FILE),
    "LIMBIC_SKIP_PIP_INSTALL": "1",
    "TORCH_TPU_DEFER_AND_FUSE": "1",
    "PYTHONUNBUFFERED": "1",
})

log_fp = open(LOG_FILE, "a", encoding="utf-8")
proc = subprocess.Popen(
    ["bash", str(REPO_DIR / "scripts" / "run_colab_tpu_v5e_4bit.sh")],
    cwd=str(REPO_DIR),
    env=env,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
PID_FILE.write_text(str(proc.pid), encoding="utf-8")
print(f"🚀 LIMBIC-0 4-Bit TPU 백엔드 부팅 시작! (PID={proc.pid}, 로그={LOG_FILE})")
print("   샤드 스트리밍 4-bit 양자화 및 6D 체성 신경 서브스페이스 보정 로그를 실시간 출력합니다...\n")

read_offset = 0
health_ok = False
cloudflare_url = ""
t_start = time.time()

while True:
    if LOG_FILE.exists():
        text = LOG_FILE.read_text(encoding="utf-8", errors="replace")
        if len(text) > read_offset:
            chunk = text[read_offset:]
            read_offset = len(text)
            print(chunk, end="", flush=True)

    if proc.poll() is not None:
        raise RuntimeError(f"❌ 서버 프로세스가 비정상 종료되었습니다 (exit code={proc.returncode}). 위 로그를 확인해 주세요.")

    if TUNNEL_URL_FILE.exists() and not cloudflare_url:
        candidate = TUNNEL_URL_FILE.read_text(encoding="utf-8").strip()
        if candidate.startswith("https://"):
            cloudflare_url = candidate

    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2.5) as resp:
            if resp.status == 200:
                payload = json.loads(resp.read().decode("utf-8"))
                health_ok = bool(payload.get("ok"))
                cf_from_api = (payload.get("public_urls") or {}).get("cloudflare") or ""
                if cf_from_api.startswith("https://"):
                    cloudflare_url = cf_from_api
    except Exception:
        pass

    if health_ok and cloudflare_url:
        break

    time.sleep(2.0)

# 서버의 /api/client-html 엔드포인트에서 Cloudflare 주소가 자동 내장된 독립 HTML 추출
with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/client-html", timeout=5.0) as resp:
    exported_html = resp.read().decode("utf-8")
EXPORTED_HTML_PATH.write_text(exported_html, encoding="utf-8")

elapsed_min = (time.time() - t_start) / 60.0
print("\n" + "=" * 78)
print(f"🎉 [개통 완료] 결(HUMAN-01) × {MODEL_ID} (4-Bit TPU) 서버가 정상 가동 중입니다! ({elapsed_min:.1f}분 소요)")
print("=" * 78)
print(f"🌐 Cloudflare 터널 백엔드 주소 : {cloudflare_url}")
print(f"🖥️ 외부 독립 HTML 파일 경로    : {EXPORTED_HTML_PATH}")
print("   -> 내 PC / 맥 / 스마트폰 어디서든 아래 다운로드된 `limbic_external_chat.html` 파일을")
print("      더블클릭해 브라우저로 열면 코랩 밖에서 곧바로 1:1 실시간 대화가 시작됩니다!")
print("   -> 또는 GitHub 레포의 `web/limbic_chat_client.html` 파일을 열고 상단 [서버 연결] 버튼에")
print(f"      `{cloudflare_url}` 주소만 붙여넣어도 즉시 연결됩니다.")
print("=" * 78)

if AUTO_DOWNLOAD_EXTERNAL_HTML:
    try:
        from google.colab import files  # type: ignore
        files.download(str(EXPORTED_HTML_PATH))
        print("📥 브라우저로 `limbic_external_chat.html` 자동 다운로드를 요청했습니다!")
    except Exception as exc:
        print(f"ℹ️ 자동 다운로드 건너뜀 ({exc}) — 좌측 파일 탐색기에서 {EXPORTED_HTML_PATH} 파일을 우클릭해 다운로드하세요.")

In [ ]:
#@title [4단계] 실시간 서버 상태 · 결(`HUMAN-01`) 바이탈 & TPU HBM 텔레메트리 확인 (언제든 실행 가능)
import json
from pathlib import Path
import urllib.request

PORT = 8000  #@param {type:"integer"}
TAIL_LOG_LINES = 35  #@param {type:"integer"}

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=4.0) as r:
        health = json.loads(r.read().decode("utf-8"))
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/souls", timeout=4.0) as r:
        souls_info = json.loads(r.read().decode("utf-8"))

    cf_url = (health.get("public_urls") or {}).get("cloudflare", "(미감지)")
    print(f"✅ 서버 상태: LIVE | 모델: {health.get('model_id')} | Cloudflare URL: {cf_url}")
    for s in souls_info.get("souls", []):
        v = s.get("vitals") or {}
        hw = s.get("hardware") or {}
        print(
            f"🫀 [{s.get('soul_id')} / {s.get('label_ko')}] "
            f"HR={v.get('heart_rate_bpm', 0):.1f} bpm | "
            f"ATP={v.get('cerebral_atp_mmol', 0):.2f} mM | "
            f"Cortisol={v.get('cortisol_ug_dl', 0):.1f} | "
            f"Adenosine={v.get('adenosine_sleep_pressure', 0):.2f} | "
            f"가속기={hw.get('gpu_name', 'TPU')} ({hw.get('gpu_vram_used_mb', 0):.0f}/{hw.get('gpu_vram_total_mb', 0):.0f} MB)"
        )
except Exception as exc:
    print(f"⚠️ 서버 응답 확인 실패: {exc}")

log_path = Path("/tmp/limbic_server.log")
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"\n📜 최근 서버 로그 ({min(TAIL_LOG_LINES, len(lines))}줄):")
    print("-" * 72)
    print("\n".join(lines[-TAIL_LOG_LINES:]))

In [ ]:
#@title [5단계] 코랩 외부용 HTML 다시 다운로드 또는 서버 안전 종료 (선택 실행)
import os
from pathlib import Path
import signal
import time
import urllib.request

ACTION = "download_external_html"  #@param ["download_external_html", "graceful_stop_server"]
PORT = 8000  #@param {type:"integer"}

if ACTION == "download_external_html":
    out_html = Path("/content/limbic_external_chat.html")
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/client-html", timeout=5.0) as resp:
        out_html.write_text(resp.read().decode("utf-8"), encoding="utf-8")
    print(f"✅ 최신 Cloudflare 터널 URL이 주입된 {out_html} 재생성 완료!")
    try:
        from google.colab import files  # type: ignore
        files.download(str(out_html))
    except Exception as exc:
        print(f"좌측 파일 탭에서 {out_html} 파일을 다운로드해 주세요 ({exc}).")
elif ACTION == "graceful_stop_server":
    pid_file = Path("/tmp/limbic_server.pid")
    if pid_file.exists():
        pid = int(pid_file.read_text().strip())
        print(f"🛑 서버(PID {pid})에 안전 종료 신호(SIGINT) 전송 중 (영혼/LoRA 체크포인트 저장)...")
        try:
            os.kill(pid, signal.SIGINT)
            time.sleep(3.0)
        except ProcessLookupError:
            pass
        pid_file.unlink(missing_ok=True)
        print("✅ 안전 종료 완료.")
    else:
        print("ℹ️ 실행 중인 서버 PID 파일이 없습니다.")